In [0]:
import data.utilities.common as Commonlib
import delta, yaml

from beertech_utils.data.integrity.integrity_check import EmptyDataFrameCheck
from data.utilities.medallion import Medallion
from data.utilities.medallion_layer import MedallionLayer as ML
from data.utilities.publishers import SnowflakeExternalTable
from pyspark.sql.functions import col, expr

In [0]:
# this entire command block will re-execute each time the "1. Subject Area" drop-down selection is changed to show valid configurations only
CONFIG_BASE_PATH = "../../configuration/"

subject_entries = Commonlib.list_directories(CONFIG_BASE_PATH)
dbutils.widgets.dropdown("subject_area", "", [""] + sorted(subject_entries), "1. Subject Area")
subject_area = dbutils.widgets.get("subject_area")

config_selection_path = f"{CONFIG_BASE_PATH}{subject_area}"
config_entries = Commonlib.list_files(config_selection_path, max_depth=1)
dbutils.widgets.dropdown("config_file", "", [""] + sorted(config_entries), "2. Configuration File")

In [0]:
# load configs using parameter values
config_file = dbutils.widgets.get("config_file")
config_file_path = f"{CONFIG_BASE_PATH}{subject_area}/{config_file}"

config = Commonlib.get_object_config(config_file_path)
source_config = config.get("source")
bronze_config = config.get(ML.bronze)
silver_config = config.get(ML.silver)

In [0]:
try:
    medallion = Medallion(config)

    # this should handle all watermarking
    medallion.read_table(read_layer=ML.bronze, write_layer=ML.silver)
    # write to silver
    medallion.cast_silver_dtypes_expr()
    # TODO: should this be instead applied while invoking reading_data instead ?
    medallion.df = medallion.df.filter(expr(silver_config.get("filter", "true")))
    # TODO: this cannot be forced
    medallion.drop_duplicates()
except Exception as e:
    medallion.logger.error(
        f"an error occured while processing the generic_procedure(bronze_to_silver) job. subject={subject_area}, config_file={config_file}: {e}"
    )
    raise

In [0]:
try:
    load_type = silver_config.get("load_type", "")

    if load_type == "delete_insert_by_batch":
        medallion.logger.info(
            f"delete batch data from silver started. subject={subject_area}, config_file={config_file}"
        )

        batch_field = silver_config.get("batch_field", "")
        delete_batches = medallion.df.select(col(batch_field)).distinct()
        delete_batches = delete_batches.rdd.flatMap(lambda x: x).collect()

        medallion.logger.info(f"deleted batches {delete_batches}. subject={subject_area}, config_file={config_file}")

        if spark.catalog.tableExists(medallion.table_lookup["silver"]):
            delta_table = spark.read.table(medallion.table_lookup["silver"])
            delete_rows = delta_table.delete(col(batch_field).isin(delete_batches))

        medallion.logger.info(f"write to silver started. subject={subject}, job={job_name}")
        medallion.write_to_delta(load_type="append", layer="silver")

    elif medallion.df.count() > 0:
        medallion.write_to_delta(load_type=load_type, layer=ML.silver)

    else:
        medallion.logger.info("Dataframe is empty, no records written to silver layer")

    medallion.logger.info(
        f"successfully processed data for silver table: {medallion.table_lookup['silver']}. subject={subject_area}, config_file={config_file}"
    )

    # Cleanup
    medallion.df.unpersist()
except Exception as e:
    medallion.logger.error(
        f"an error occured while writing to silver via generic_procedure(bronze_to_silver). subject={subject_area}, config_file={config_file}: {e}"
    )
    raise

In [0]:
try:
    # publish to external stage - snowflake
    if config.get("publish_to_sf", False):
        pub = SnowflakeExternalTable(medallion.catalog, medallion.schema, medallion.name)
        pub.publish_ext_table()
except Exception as e:
    medallion.logger.error(
        f"An error occured while publishing to snowflake via generic_procedure(bronze_to_silver). subject={subject_area}, config_file={config_file}: {e}"
    )
    raise

In [0]:
# compare legacy and modern datasets
try:
    medallion.compare_legacy_and_modern(
        config.get("silver_name", config.get("name")),
        config["comparison_check"]["legacy_query"],
        config["comparison_check"]["modern_query"],
        medallion.key_columns,
        config.get("comparison_check", {}).get("completeness_lower_bound"),
        config.get("comparison_check", {}).get("equivalency_lower_bound"),
        config.get("comparison_check", {}).get("col_exclusions"),
        config.get("comparison_check", {}).get("auto_resolve_col_diffs"),
    )
except KeyError as e:
    medallion.logger.warning(
        "Skipping comparison of legacy and modern because the required config is missing or invalid."
    )